In [1]:
import ccxt
import pandas as pd
import pandas_ta as ta
import numpy as np
import time
import os
import plotly.graph_objects as go
import plotly.express as px

In [2]:
# --- 1. ดึงข้อมูลผ่าน CCXT (ใช้ Binance เป็นตัวอย่าง) ---
# ตั้งชื่อไฟล์ที่เราจะใช้เก็บข้อมูล
filename = 'btc_usdt_4h_3years.csv'

# --- ลอจิกตรวจสอบไฟล์: มีไฟล์อยู่แล้วโหลดเลย / ไม่มีค่อยดึงใหม่ ---
if os.path.exists(filename):
    print(f"🔍 พบไฟล์ '{filename}' ในระบบ")
    print("กำลังโหลดข้อมูลจากไฟล์... (ข้ามขั้นตอนการต่อ API)")
    
    # โหลดไฟล์ CSV และตั้งให้คอลัมน์ Timestamp กลับมาเป็น Index (เวลา) เหมือนเดิม
    df = pd.read_csv(filename, index_col='Timestamp', parse_dates=True)
    
    print(f"\n✅ โหลดข้อมูลสำเร็จ! จำนวนทั้งหมด: {len(df):,} แท่ง")
    
else:
    print(f"ไม่พบไฟล์ '{filename}' กำลังเชื่อมต่อ Binance เพื่อดึงข้อมูลย้อนหลัง 5 ปี...")
    
    exchange = ccxt.binance({'enableRateLimit': True})
    symbol = 'BTC/USDT'
    timeframe = '4h'
    years_to_fetch = 3
    
    now = exchange.milliseconds()
    five_years_ms = int(years_to_fetch * 365.25 * 24 * 60 * 60 * 1000)
    since = now - five_years_ms
    
    all_bars = []
    
    while since < now:
        try:
            bars = exchange.fetch_ohlcv(symbol, timeframe=timeframe, since=since, limit=1000)
            if not bars:
                break
            
            all_bars.extend(bars)
            since = bars[-1][0] + 1
            
            last_date = pd.to_datetime(bars[-1][0], unit='ms').strftime('%Y-%m-%d %H:%M:%S')
            print(f"ดึงข้อมูลสะสมแล้ว {len(all_bars):,} แท่ง... (ข้อมูลล่าสุดถึง: {last_date})")
            
            time.sleep(0.5) # พักเบรกให้ API ปลอดภัยจากการถูกแบน
            
        except Exception as e:
            print(f"เกิดข้อผิดพลาด: {e} -> จะลองดึงใหม่ในอีก 5 วินาที...")
            time.sleep(5)
            continue
            
    # แปลงข้อมูลเป็น DataFrame
    df = pd.DataFrame(all_bars, columns=['Timestamp', 'Open', 'High', 'Low', 'Close', 'Volume'])
    df['Timestamp'] = pd.to_datetime(df['Timestamp'], unit='ms')
    df.set_index('Timestamp', inplace=True)
    df = df[~df.index.duplicated(keep='first')]
    
    # --- สำคัญ: บันทึกข้อมูลลงไฟล์ CSV สำหรับใช้รอบหน้า ---
    df.to_csv(filename)
    print(f"\n✅ ดึงข้อมูลและบันทึกลงไฟล์ '{filename}' สำเร็จ! จำนวนทั้งหมด: {len(df):,} แท่ง")

print(f"ข้อมูลเริ่มต้นตั้งแต่: {df.index[0]} ถึง {df.index[-1]}")

🔍 พบไฟล์ 'btc_usdt_4h_3years.csv' ในระบบ
กำลังโหลดข้อมูลจากไฟล์... (ข้ามขั้นตอนการต่อ API)

✅ โหลดข้อมูลสำเร็จ! จำนวนทั้งหมด: 6,574 แท่ง
ข้อมูลเริ่มต้นตั้งแต่: 2023-03-16 04:00:00 ถึง 2026-03-15 16:00:00


In [3]:
# --- 2. ตั้งค่าพารามิเตอร์ (เหมือนใน Pine Script) ---
lookback = 4
atr_len = 14
atr_mult = 1.5
vol_len = 20
tolerance_bars = 2

In [4]:
# --- 3. คำนวณค่าต่างๆ แบบรวดเดียว (Vectorized) ---
# คำนวณ ATR
df['ATR'] = ta.atr(df['High'], df['Low'], df['Close'], length=atr_len)

# หา Highest High และ Lowest Low ย้อนหลัง 4 แท่ง
df['HH'] = df['High'].rolling(window=lookback).max()
df['LL'] = df['Low'].rolling(window=lookback).min()
df['BoxRange'] = df['HH'] - df['LL']

# คำนวณ Volume เงื่อนไข
df['AvgVol_4'] = df['Volume'].rolling(window=lookback).mean()
df['SMA_Vol_20'] = df['Volume'].rolling(window=vol_len).mean()
df['VolCondition'] = df['AvgVol_4'] < df['SMA_Vol_20']

# เงื่อนไขเริ่มต้นของการเกิดกรอบสะสม
df['IsConsolidating'] = (df['BoxRange'] <= (df['ATR'] * atr_mult)) & df['VolCondition']

#คำนวณ SMA 200 จาก DataFrame ตัวเต็ม (df) ก่อน ---
# (ต้องคำนวณจาก df ตัวเต็ม เพื่อให้มีข้อมูลย้อนหลังพอสำหรับ 200 แท่ง)S
df['SMA_200'] = df['Close'].rolling(window=200).mean()

In [5]:
# --- 4. ตัวแปรเก็บสถานะการวาดกล่อง ---
in_zone = False
upper_limit = np.nan
lower_limit = np.nan
bars_outside = 0

df['InBox'] = False
df['BoxTop'] = np.nan
df['BoxBottom'] = np.nan

In [6]:
# --- 5. ชุดโค้ดตรวจสอบแบบแท่งต่อแท่งเต็มรูปแบบ (Iterative State Machine with Back-population) ---
# เซ็ตค่าเริ่มต้นใหม่
in_zone = False
upper_limit = np.nan
lower_limit = np.nan
bars_outside = 0

df['InBox'] = False
df['BoxTop'] = np.nan
df['BoxBottom'] = np.nan

# --- ลอจิกการตรวจสอบแบบแท่งต่อแท่ง (เดินหน้าอย่างเดียว ไม่ย้อนหลัง) ---
for i in range(len(df)):
    if pd.isna(df['ATR'].iloc[i]):
        continue
        
    current_close = df['Close'].iloc[i] 
    
    if in_zone:
        if current_close > upper_limit or current_close < lower_limit:
            bars_outside += 1
        else:
            bars_outside = 0
            
        if bars_outside > tolerance_bars:
            in_zone = False # จบกล่อง
            bars_outside = 0
        else:
            # ยืดกล่องน้ำเงินต่อไป
            df.iat[i, df.columns.get_loc('InBox')] = True
            df.iat[i, df.columns.get_loc('BoxTop')] = upper_limit
            df.iat[i, df.columns.get_loc('BoxBottom')] = lower_limit
            
    if not in_zone:
        # เจอแท่งที่ 4 ที่เข้าเงื่อนไข (เริ่มเข้าโซน)
        if df['IsConsolidating'].iloc[i]:
            in_zone = True
            upper_limit = df['HH'].iloc[i]
            lower_limit = df['LL'].iloc[i]
            bars_outside = 0
            
            # บันทึกกล่องน้ำเงินแท่งแรก
            df.iat[i, df.columns.get_loc('InBox')] = True
            df.iat[i, df.columns.get_loc('BoxTop')] = upper_limit
            df.iat[i, df.columns.get_loc('BoxBottom')] = lower_limit

In [7]:
# --- 6. แสดงผลลัพธ์ ---
print("\nตัวอย่างข้อมูลช่วงที่เกิด Consolidation 10 แท่งล่าสุด:")
consolidation_data = df[df['InBox'] == True][['Close', 'BoxTop', 'BoxBottom']]
print(consolidation_data.head(10))


ตัวอย่างข้อมูลช่วงที่เกิด Consolidation 10 แท่งล่าสุด:
                        Close    BoxTop  BoxBottom
Timestamp                                         
2023-03-19 08:00:00  27160.98  27449.95   26810.00
2023-03-19 12:00:00  27546.62  27449.95   26810.00
2023-03-19 16:00:00  28255.87  27449.95   26810.00
2023-03-21 00:00:00  27787.29  28258.58   27461.35
2023-03-21 04:00:00  27505.25  28258.58   27461.35
2023-03-21 08:00:00  28040.13  28258.58   27461.35
2023-03-21 12:00:00  28041.06  28258.58   27461.35
2023-03-21 16:00:00  28126.84  28258.58   27461.35
2023-03-21 20:00:00  28105.47  28258.58   27461.35
2023-03-22 00:00:00  28154.14  28258.58   27461.35


In [8]:
plot_df = df.tail(7000).copy() # ดึงมา 500 แท่งล่าสุด

fig = go.Figure(data=[go.Candlestick(
    x=plot_df.index, 
    open=plot_df['Open'], 
    high=plot_df['High'], 
    low=plot_df['Low'], 
    close=plot_df['Close'], 
    name='Price'
)])

# --- 3. เพิ่มเส้น SMA 200 (พล็อตเป็นเส้น Scatter) ---
fig.add_trace(go.Scatter(
    x=plot_df.index,
    y=plot_df['SMA_200'],
    mode='lines',
    line=dict(color='orange', width=2),
    name='SMA 200'
))

plot_df['BoxGroup'] = (plot_df['InBox'] != plot_df['InBox'].shift()).cumsum()
boxes = plot_df[plot_df['InBox'] == True].groupby('BoxGroup')

for name, group in boxes:
    if len(group) >= 1:
        # 1. ข้อมูลของกล่องสีน้ำเงิน (เริ่มจากแท่งที่ 4 เป็นต้นไป)
        start_time_blue = group.index[0]
        end_time_blue = group.index[-1]
        top = group['BoxTop'].iloc[0]
        bottom = group['BoxBottom'].iloc[0]
        
        # 2. คำนวณหาจุดเริ่มต้นของกล่องสีแดง (ย้อนกลับไป lookback-1 แท่ง)
        # หาตำแหน่ง index (บรรทัดที่เท่าไหร่) ของแท่งที่เริ่มกล่องน้ำเงิน
        idx_blue_start = plot_df.index.get_loc(start_time_blue)
        # ถอยหลังไป 3 แท่ง (หรือตามค่า lookback)
        idx_red_start = max(0, idx_blue_start - lookback + 1)
        start_time_red = plot_df.index[idx_red_start]

        # --- วาดกล่องสีแดง (ช่วงสะสม 4 แท่งแรก) ---
        fig.add_shape(
            type="rect",
            x0=start_time_red, y0=bottom, 
            x1=start_time_blue, y1=top,
            line=dict(color="rgba(255, 50, 50, 1)", width=2),   # ขอบสีแดง
            fillcolor="rgba(255, 50, 50, 0.2)",                 # พื้นหลังสีแดงโปร่งแสง
            layer="below"
        )

        # --- วาดกล่องสีน้ำเงิน (ช่วงที่ราคายังอยู่ในกรอบ รอ Breakout) ---
        fig.add_shape(
            type="rect",
            x0=start_time_blue, y0=bottom, 
            x1=end_time_blue, y1=top,
            line=dict(color="rgba(41, 98, 255, 1)", width=2),   # ขอบสีน้ำเงิน
            fillcolor="rgba(41, 98, 255, 0.2)",                 # พื้นหลังสีน้ำเงินโปร่งแสง
            layer="below"
        )

fig.update_layout(
    title='Crypto Consolidation (Red = Setup, Blue = Waiting for Breakout)',
    yaxis_title='Price (USDT)', 
    xaxis_title='Time',
    xaxis_rangeslider_visible=False, 
    template='plotly_dark', 
    height=600
)

fig.show()

In [33]:
# เตรียม List ว่างเพื่อเก็บข้อมูลสถิติของแต่ละกล่อง
research_data = []

# ใช้ตัวแปร boxes จากขั้นตอนก่อนหน้าที่เรา Group ไว้แล้ว
for name, group in boxes:
    if len(group) < 1: 
        continue

    # 1. กำหนดจุดเวลาของกล่อง
    box_start_time = group.index[0]
    box_end_time = group.index[-1]

    #print(box_start_time)
    #print(box_end_time)
  
    
    # 2. คำนวณค่ากึ่งกลางและระยะของกล่อง
    top = group['BoxTop'].iloc[0]
    bottom = group['BoxBottom'].iloc[0]
    box_center = (top + bottom) / 2
    box_range = top - bottom
    
    # ดึงค่า SMA 200 ณ แท่งสุดท้ายของกล่อง (ตอนเบรกเอาท์)
    sma_val = df.loc[box_start_time, 'SMA_200']
    
    # ถ้ากราฟช่วงแรกยังไม่มี SMA 200 ให้ข้ามกล่องนี้ไปก่อน
    if pd.isna(sma_val): 
        continue
        
    # 3. กำหนดทิศทาง (Trend Direction) และจุดตัดขาดทุน (Stop Level)
    # ถ้าราคากึ่งกลาง อยู่เหนือ SMA 200 = มองขึ้น (Long) / ต่ำกว่า = มองลง (Short)
    is_uptrend = box_center > sma_val
    
    if is_uptrend:
        direction_label = 'Long'
        stop_level = box_center - box_range  # สวนทาง 1 กรอบ = ตัดขาดทุน
    else:
        direction_label = 'Short'
        stop_level = box_center + box_range  # สวนทาง 1 กรอบ = ตัดขาดทุน
        
    # 4. สแกนราคาในอนาคต (นับตั้งแต่ตอนกรอบเริ่มสร้าง)
    # ตัด DataFrame เอาเฉพาะข้อมูลตั้งแต่อนาคตเป็นต้นไป โดยไม่ต้องข้ามแท่งแรก (.iloc[1:])
    future_df = df.loc[box_start_time:].iloc[1:]
    #print(future_df)  

    max_favorable_distance = 0  # ระยะที่วิ่งไปถูกทางได้ไกลที่สุด
    hit_stop_loss = False       # โดนจุดติดลบหรือไม่
    hit_break_even = False
    trigger_be = False
    max3d = False
    is_triggered = False        # รอดูว่าราคากลับมาแตะตรงกลางกรอบไหม
    is_brakeout = False         # รอดูราคาทะลุกรอบก่อนนับจำนวนแท่งเทียน
    
    # ลอจิกการจำลองเดินหน้าไปทีละแท่ง
    if is_uptrend:
        followCandel = 0
        highest_seen = box_center
        for idx, row in future_df.iterrows():
            # เช็คว่าราคาย้อนกลับมาทับกลางกรอบ (Trigger รับออเดอร์) หรือยัง
            if not is_triggered:
                if row['Low'] <= box_center:
                    is_triggered = True  # กลับมาที่ศูนย์กลางแล้ว
                elif idx  == box_end_time:
                    break #ถ้าหลุดกรอบให้หยุดแล้วไปเริ่มกรองใหม่
                else:
                    # ถ้าราคายังไม่แตะกลางกรอบ ให้ข้ามการทำงานในแท่งนี้ไป
                    continue

            # เมื่อถูก Trigger แล้ว ถึงจะเริ่มเช็คจุดกำไร และขยับ SL
            # อัปเดตระยะที่วิ่งไปทำกำไรได้สูงสุด
            if row['High'] > highest_seen:
                highest_seen = row['High']
                if (highest_seen - box_center) > (box_range/2):
                    trigger_be = True
            
            if row['Close'] > top:
                is_brakeout = True

            if is_brakeout :
                followCandel += 1 #เริ่มนับแท่งเทียน
              
            # ถ้าราคา Low ตวัดลงมาโดนจุด Stop Level (ตรงข้าม 1 กรอบ) -> จบรอบ
            if row['Low'] <= (box_center+box_range) and trigger_be:
                hit_break_even = True
                break
            elif row['Low'] <= stop_level :
                hit_stop_loss = True
                break
           
            if followCandel == 18:
                max3d = True
                break
                
        max_favorable_distance = highest_seen - box_center
        # ระยะที่บันทึกผลจริง: ถ้าโดน Stop Loss ให้บันทึกติดลบ 1 กรอบ
        realized_distance = -box_range if hit_stop_loss else (highest_seen - box_center)
        
    else: # ฝั่ง Short (Downtrend)
        followCandel = 0
        lowest_seen = box_center
        for idx, row in future_df.iterrows():
            
            # เช็คว่าราคาได้ทำจุดสูงสุดแตะกลางกรอบ (Trigger รับออเดอร์) หรือยัง
            if not is_triggered:
                if row['High'] >= box_center:
                    is_triggered = True
                elif idx  == box_end_time:
                    break #ถ้าหลุดกรอบให้หยุดแล้วไปเริ่มกรองใหม่
                else:
                    continue


            if row['Low'] < lowest_seen:
                lowest_seen = row['Low']
                if (box_center - lowest_seen) > (box_range/2) :
                    trigger_be = True
                    
            if row['Close'] < bottom:
                is_brakeout = True

            if is_brakeout :
                followCandel += 1 #เริ่มนับแท่งเทียน  
            
            if row['High'] >= (box_center+box_range) and trigger_be:
                hit_break_even = True
                break
            elif row['High'] >= stop_level  :
                hit_stop_loss = True
                break
                
            if followCandel == 18:
                max3d = True
                break
                
        max_favorable_distance = box_center - lowest_seen
        realized_distance = -box_range if hit_stop_loss else (box_center - lowest_seen)
        
    # 5. เก็บข้อมูลทั้งหมดลง List (กรองเฉพาะกล่องที่ถูก Trigger มารับออเดอร์จริงๆ เท่านั้น)
    if is_triggered:
        research_data.append({
            'Start_Time': box_start_time,
            'End_Time': box_end_time,
            'Trend': direction_label,
            'Box_Center': round(box_center, 2),
            'Box_Range': round(box_range, 2),
            'Max_Profit_Dist': round(max_favorable_distance, 2), # ระยะบวกมากสุดก่อนโดน SL
            'Hit_SL': hit_stop_loss,                             # ชนกรอบฝั่งตรงข้ามไหม (True/False)
            'Final_Result': round(realized_distance, 2)          # ผลลัพธ์สุดท้าย (บวก หรือ ติดลบ)
        })

# แปลง List เป็น DataFrame เพื่อง่ายต่อการวิเคราะห์
research_df = pd.DataFrame(research_data)

# สร้างคอลัมน์คิดเป็นอัตราส่วน Reward / Risk (RR Ratio) (ใส่กัน error หากไม่มีข้อมูลผ่านเงื่อนไขเลย)
if not research_df.empty:
    research_df['RR_Ratio'] = round(research_df['Final_Result'] / research_df['Box_Range'], 2)

# แสดงผลข้อมูลวิจัย 10 รอบล่าสุด
#print(research_df.tail(100).to_string(index=False))


In [ ]:
""" # เตรียม List ว่างเพื่อเก็บข้อมูลสถิติของแต่ละกล่อง
research_data = []

# ใช้ตัวแปร boxes จากขั้นตอนก่อนหน้าที่เรา Group ไว้แล้ว
for name, group in boxes:
    if len(group) < 1: 
        continue

    # 1. กำหนดจุดเวลาของกล่อง
    box_start_time = group.index[0]
    box_end_time = group.index[-1]
    
    # 2. คำนวณค่ากึ่งกลางและระยะของกล่อง
    top = group['BoxTop'].iloc[0]
    bottom = group['BoxBottom'].iloc[0]
    box_center = (top + bottom) / 2
    box_range = top - bottom
    
    # ดึงค่า SMA 200 ณ แท่งสุดท้ายของกล่อง (ตอนเบรกเอาท์)
    #sma_val = df.loc[box_end_time, 'SMA_200']
    sma_val = df.loc[box_start_time, 'SMA_200']
    # ถ้ากราฟช่วงแรกยังไม่มี SMA 200 ให้ข้ามกล่องนี้ไปก่อน
    if pd.isna(sma_val): 
        continue
        
    # 3. กำหนดทิศทาง (Trend Direction) และจุดตัดขาดทุน (Stop Level)
    # ถ้าราคากึ่งกลาง อยู่เหนือ SMA 200 = มองขึ้น (Long) / ต่ำกว่า = มองลง (Short)
    is_uptrend = box_center > sma_val
    
    if is_uptrend:
        direction_label = 'Long'
        stop_level = box_center - box_range  # สวนทาง 1 กรอบ = ตัดขาดทุน
    else:
        direction_label = 'Short'
        stop_level = box_center + box_range  # สวนทาง 1 กรอบ = ตัดขาดทุน
        
    # 4. สแกนราคาในอนาคต (นับจากแท่งถัดจากที่เบรกเอาท์ไปแล้ว)
    # ตัด DataFrame เอาเฉพาะข้อมูลตั้งแต่อนาคตเป็นต้นไป
    future_df = df.loc[box_start_time:].iloc[1:]
    
    #print(future_df)   



    max_favorable_distance = 0  # ระยะที่วิ่งไปถูกทางได้ไกลที่สุด
    hit_stop_loss = False       # โดนจุดติดลบหรือไม่
    hit_break_even = False
    trigger_be = False
    max3d = False
    
    


    # ลอจิกการจำลองเดินหน้าไปทีละแท่ง
    if is_uptrend:
        followCandel = 0
        highest_seen = box_center
        for idx, row in future_df.iterrows():
            # อัปเดตระยะที่วิ่งไปทำกำไรได้สูงสุด
            if row['High'] > highest_seen:
                highest_seen = row['High']
                if (highest_seen - box_center) > (box_range/2) :
                    trigger_be = True
                
            followCandel =+ 1
            # ถ้าราคา Low ตวัดลงมาโดนจุด Stop Level (ตรงข้าม 1 กรอบ) -> จบรอบ

            if row['Low'] <= (box_center+box_range) and trigger_be:
                hit_break_even = True
                break
            elif row['Low'] <= stop_level :
                hit_stop_loss = True
                break

           
            if followCandel == 18:
                max3d = True
                break
                
        max_favorable_distance = highest_seen - box_center
        # ระยะที่บันทึกผลจริง: ถ้าโดน Stop Loss ให้บันทึกติดลบ 1 กรอบ
        realized_distance = -box_range if hit_stop_loss else (highest_seen - box_center)
        
    else: # ฝั่ง Short (Downtrend)
        followCandel = 0
        lowest_seen = box_center
        for idx, row in future_df.iterrows():
            if row['Low'] < lowest_seen:
                lowest_seen = row['Low']
                if (box_center - lowest_seen) > (box_range/2) :
                    trigger_be = True
            followCandel =+ 1    
            if row['High'] >= (box_center+box_range) and trigger_be:
                hit_break_even = True
                break
            elif row['High'] >= stop_level  :
                hit_stop_loss = True
                break
            if followCandel == 18:
                max3d = True
                break
                
        max_favorable_distance = box_center - lowest_seen
        realized_distance = -box_range if hit_stop_loss else (box_center - lowest_seen)
        
    # 5. เก็บข้อมูลทั้งหมดลง List
    research_data.append({
        'Start_Time': box_start_time,
        'End_Time': box_end_time,
        'Trend': direction_label,
        'Box_Center': round(box_center, 2),
        'Box_Range': round(box_range, 2),
        'Max_Profit_Dist': round(max_favorable_distance, 2), # ระยะบวกมากสุดก่อนโดน SL
        'Hit_SL': hit_stop_loss,                             # ชนกรอบฝั่งตรงข้ามไหม (True/False)
        'Final_Result': round(realized_distance, 2)          # ผลลัพธ์สุดท้าย (บวก หรือ ติดลบ)
    })

# แปลง List เป็น DataFrame เพื่อง่ายต่อการวิเคราะห์
research_df = pd.DataFrame(research_data)

# สร้างคอลัมน์คิดเป็นอัตราส่วน Reward / Risk (RR Ratio)
research_df['RR_Ratio'] = round(research_df['Final_Result'] / research_df['Box_Range'], 2)

# แสดงผลข้อมูลวิจัย 10 รอบล่าสุด
#print(research_df.tail(100).to_string(index=False)) """

In [25]:
# --- พล็อตจุด Entry จาก research_df ลงบนกราฟแท่งเทียน ---

fig = go.Figure()

# 1. วาดแท่งเทียนทั้งหมด
fig.add_trace(go.Candlestick(
    x=df.index,
    open=df['Open'], high=df['High'],
    low=df['Low'], close=df['Close'],
    name='Price',
    increasing_line_color='#26a69a',
    decreasing_line_color='#ef5350'
))

# 2. วาดเส้น SMA 200
fig.add_trace(go.Scatter(
    x=df.index, y=df['SMA_200'],
    mode='lines',
    line=dict(color='orange', width=1.5),
    name='SMA 200'
))


# 3. แยก research_df เป็น Long / Short
long_df  = research_df[research_df['Trend'] == 'Long'].copy()
short_df = research_df[research_df['Trend'] == 'Short'].copy()

# คำนวณระดับราคา Max Profit สูงสุดที่เคยแตะ
# Long: ราคาวิ่งขึ้นไปสูงสุด = Box_Center + Max_Profit_Dist
# Short: ราคาวิ่งลงไปต่ำสุด = Box_Center - Max_Profit_Dist
long_df['Max_Price_Level']  = long_df['Box_Center']  + long_df['Max_Profit_Dist']
short_df['Max_Price_Level'] = short_df['Box_Center'] - short_df['Max_Profit_Dist']

# 4. วาดจุด Long Entry (สีเขียว ▲)
fig.add_trace(go.Scatter(
    x=long_df['End_Time'], y=long_df['Box_Center'],
    mode='markers',
    marker=dict(symbol='triangle-up', color='#00e676', size=10,
                line=dict(color='white', width=0.5)),
    name='Long Entry',
    customdata=long_df[['Box_Range', 'Max_Profit_Dist', 'Hit_SL', 'Final_Result', 'RR_Ratio']].values,
    hovertemplate=(
        '<b>LONG Entry</b><br>Time: %{x}<br>Box Center: %{y:,.2f}<br>'
        'Box Range: %{customdata[0]:,.2f}<br>Max Profit Dist: %{customdata[1]:,.2f}<br>'
        'Hit SL: %{customdata[2]}<br>Final Result: %{customdata[3]:,.2f}<br>'
        'RR Ratio: %{customdata[4]:.2f}<extra></extra>'
    )
))

# 5. วาดจุด Short Entry (สีแดง ▼)
fig.add_trace(go.Scatter(
    x=short_df['End_Time'], y=short_df['Box_Center'],
    mode='markers',
    marker=dict(symbol='triangle-down', color='#ff1744', size=10,
                line=dict(color='white', width=0.5)),
    name='Short Entry',
    customdata=short_df[['Box_Range', 'Max_Profit_Dist', 'Hit_SL', 'Final_Result', 'RR_Ratio']].values,
    hovertemplate=(
        '<b>SHORT Entry</b><br>Time: %{x}<br>Box Center: %{y:,.2f}<br>'
        'Box Range: %{customdata[0]:,.2f}<br>Max Profit Dist: %{customdata[1]:,.2f}<br>'
        'Hit SL: %{customdata[2]}<br>Final Result: %{customdata[3]:,.2f}<br>'
        'RR Ratio: %{customdata[4]:.2f}<extra></extra>'
    )
))

# 6. ⭐ จุด Max Profit ของ Long (จุดสูงสุดที่ราคาเคยวิ่งขึ้นไปถึง)
fig.add_trace(go.Scatter(
    x=long_df['End_Time'], y=long_df['Max_Price_Level'],
    mode='markers',
    marker=dict(symbol='star', color='#ffff00', size=9,
                line=dict(color='#cccc00', width=0.5)),
    name='Long Max Profit ⭐',
    customdata=long_df[['Box_Center', 'Max_Profit_Dist', 'RR_Ratio']].values,
    hovertemplate=(
        '<b>⭐ Long Max Profit</b><br>Time: %{x}<br>'
        'Max Price: %{y:,.2f}<br>'
        'Entry (Center): %{customdata[0]:,.2f}<br>'
        'Distance: %{customdata[1]:,.2f}<br>'
        'RR Ratio: %{customdata[2]:.2f}<extra></extra>'
    )
))

# 7. ⭐ จุด Max Profit ของ Short (จุดต่ำสุดที่ราคาเคยวิ่งลงไปถึง)
fig.add_trace(go.Scatter(
    x=short_df['End_Time'], y=short_df['Max_Price_Level'],
    mode='markers',
    marker=dict(symbol='star', color='#ff9900', size=9,
                line=dict(color='#cc7700', width=0.5)),
    name='Short Max Profit ⭐',
    customdata=short_df[['Box_Center', 'Max_Profit_Dist', 'RR_Ratio']].values,
    hovertemplate=(
        '<b>⭐ Short Max Profit</b><br>Time: %{x}<br>'
        'Min Price: %{y:,.2f}<br>'
        'Entry (Center): %{customdata[0]:,.2f}<br>'
        'Distance: %{customdata[1]:,.2f}<br>'
        'RR Ratio: %{customdata[2]:.2f}<extra></extra>'
    )
))


plot_df['BoxGroup'] = (plot_df['InBox'] != plot_df['InBox'].shift()).cumsum()
boxes = plot_df[plot_df['InBox'] == True].groupby('BoxGroup')

for name, group in boxes:
    if len(group) >= 1:
        # 1. ข้อมูลของกล่องสีน้ำเงิน (เริ่มจากแท่งที่ 4 เป็นต้นไป)
        start_time_blue = group.index[0]
        end_time_blue = group.index[-1]
        top = group['BoxTop'].iloc[0]
        bottom = group['BoxBottom'].iloc[0]
        
        # 2. คำนวณหาจุดเริ่มต้นของกล่องสีแดง (ย้อนกลับไป lookback-1 แท่ง)
        # หาตำแหน่ง index (บรรทัดที่เท่าไหร่) ของแท่งที่เริ่มกล่องน้ำเงิน
        idx_blue_start = plot_df.index.get_loc(start_time_blue)
        # ถอยหลังไป 3 แท่ง (หรือตามค่า lookback)
        idx_red_start = max(0, idx_blue_start - lookback + 1)
        start_time_red = plot_df.index[idx_red_start]

        # --- วาดกล่องสีแดง (ช่วงสะสม 4 แท่งแรก) ---
        fig.add_shape(
            type="rect",
            x0=start_time_red, y0=bottom, 
            x1=start_time_blue, y1=top,
            line=dict(color="rgba(255, 50, 50, 1)", width=2),   # ขอบสีแดง
            fillcolor="rgba(255, 50, 50, 0.2)",                 # พื้นหลังสีแดงโปร่งแสง
            layer="below"
        )

        # --- วาดกล่องสีน้ำเงิน (ช่วงที่ราคายังอยู่ในกรอบ รอ Breakout) ---
        fig.add_shape(
            type="rect",
            x0=start_time_blue, y0=bottom, 
            x1=end_time_blue, y1=top,
            line=dict(color="rgba(41, 98, 255, 1)", width=2),   # ขอบสีน้ำเงิน
            fillcolor="rgba(41, 98, 255, 0.2)",                 # พื้นหลังสีน้ำเงินโปร่งแสง
            layer="below"
        )

fig.update_layout(
    title='BTC/USDT 4H — Entry (▲▼) และจุด Max Profit (⭐) จาก research_df',
    template='plotly_dark',
    xaxis_rangeslider_visible=False,
    height=700,
    legend=dict(orientation='h', yanchor='bottom', y=1.02)
)
fig.show()

print(f"✅ แสดงจุดทั้งหมด {len(research_df)} จุด | Long: {len(long_df)} | Short: {len(short_df)}")


✅ แสดงจุดทั้งหมด 328 จุด | Long: 166 | Short: 162


In [ ]:
# เตรียม List ว่างเพื่อเก็บข้อมูลสถิติของแต่ละกล่อง
research_data = []

# ใช้ตัวแปร boxes จากขั้นตอนก่อนหน้าที่เรา Group ไว้แล้ว
for name, group in boxes:
    if len(group) < 1: 
        continue

    # 1. กำหนดจุดเวลาของกล่อง
    box_start_time = group.index[0]
    box_end_time = group.index[-1]
    
    # 2. คำนวณค่ากึ่งกลางและระยะของกล่อง
    top = group['BoxTop'].iloc[0]
    bottom = group['BoxBottom'].iloc[0]
    box_center = (top + bottom) / 2
    box_range = top - bottom
    
    # ดึงค่า SMA 200 ณ แท่งสุดท้ายของกล่อง
    sma_val = df.loc[box_start_time, 'SMA_200']
    
    if pd.isna(sma_val): 
        continue
        
    # 3. กำหนดทิศทางและ Stop Level
    is_uptrend = box_center > sma_val
    
    if is_uptrend:
        direction_label = 'Long'
        stop_level = box_center - box_range 
    else:
        direction_label = 'Short'
        stop_level = box_center + box_range
        
    # 4. สแกนราคาในอนาคต
    future_df = df.loc[box_start_time:].iloc[1:]

    max_favorable_distance = 0  
    hit_stop_loss = False       
    hit_break_even = False      
    trigger_be = False
    max3d = False
    is_triggered = False        
    is_brakeout = False         
    
    # ลอจิกการจำลองเดินหน้าไปทีละแท่ง
    if is_uptrend:
        followCandel = 0
        highest_seen = box_center
        for idx, row in future_df.iterrows():
            if not is_triggered:
                if row['Low'] <= box_center:
                    is_triggered = True  
                elif idx  == box_end_time:
                    break 
                else:
                    continue

            # เลื่อนหน้าทุน (Break Even) เมื่อวิ่งไปไกล 1 เท่าของระยะกล่อง
            if row['High'] > highest_seen:
                highest_seen = row['High']
                if (highest_seen - box_center) >= box_range:
                    trigger_be = True
            
            if row['Close'] > top:
                is_brakeout = True

            if is_brakeout:
                followCandel += 1 
              
            # ถ้าราคา Low ตวัดลงมาชนจุดเข้า ณ กลางกรอบ (กั้นหน้าทุน) -> จบรอบคุ้มทุน
            if row['Low'] <= box_center and trigger_be:
                hit_break_even = True
                break
            # ถ้าราคา Low ตวัดลงมาโดนจุด Stop Level เริ่มต้น -> จบรอบ SL
            elif row['Low'] <= stop_level :
                hit_stop_loss = True
                break
           
            if followCandel == 18:
                max3d = True
                break
                
        max_favorable_distance = highest_seen - box_center
        
        # [ใหม่] คำนวณระยะที่ทำได้จริงตอนจบออเดอร์ (Actual Result)
        if hit_break_even:
            realized_distance = 0
        elif hit_stop_loss:
            realized_distance = -box_range
        else:
            # กรณีที่รันเทรนด์จนจบ 18 แท่ง หรือจนจบกราฟ -> ปิดสถานะที่ราคาปิด (Close) ของแท่งที่ออก 
            realized_distance = row['Close'] - box_center
        
    else: # ฝั่ง Short (Downtrend)
        followCandel = 0
        lowest_seen = box_center
        for idx, row in future_df.iterrows():
            if not is_triggered:
                if row['High'] >= box_center:
                    is_triggered = True
                elif idx  == box_end_time:
                    break 
                else:
                    continue

            # เลื่อนหน้าทุน (Break Even) เมื่อลงไปไกล 1 เท่าของระยะกล่อง
            if row['Low'] < lowest_seen:
                lowest_seen = row['Low']
                if (box_center - lowest_seen) >= box_range :
                    trigger_be = True
                    
            if row['Close'] < bottom:
                is_brakeout = True

            if is_brakeout:
                followCandel += 1 
            
            # ถ้าราคา High เด้งกลับขึ้นมาชนจุดเข้า ณ กลางกรอบ (กั้นหน้าทุน) -> จบรอบคุ้มทุน
            if row['High'] >= box_center and trigger_be:
                hit_break_even = True
                break
            # ถ้าราคา High ทะลุโดนจุด Stop Level เริ่มต้น -> จบรอบ SL
            elif row['High'] >= stop_level  :
                hit_stop_loss = True
                break
                
            if followCandel == 18:
                max3d = True
                break
                
        max_favorable_distance = box_center - lowest_seen
        
        # [ใหม่] คำนวณระยะที่ทำได้จริงตอนจบออเดอร์
        if hit_break_even:
            realized_distance = 0
        elif hit_stop_loss:
            realized_distance = -box_range
        else:
            # กรณีที่รันเทรนด์จนจบ 18 แท่ง หรือจนจบกราฟ -> ปิดสถานะ Short ที่ราคาปิด (Close) ของแท่งนั้น
            realized_distance = box_center - row['Close']
        
    # 5. เก็บข้อมูลทั้งหมดลง List
    if is_triggered:
        research_data.append({
            'Start_Time': box_start_time,
            'End_Time': box_end_time,
            'Trend': direction_label,
            'Box_Center': round(box_center, 2),
            'Box_Range': round(box_range, 2),
            'Max_Profit_Dist': round(max_favorable_distance, 2), # ระยะบวกสูงสุดที่เป็นไปได้
            'Hit_SL': hit_stop_loss,                             
            'Hit_BE': hit_break_even,                            
            'Actual_Result': round(realized_distance, 2)         # กำไร/ขาดทุนที่ได้จริงๆ ตอนปิดไม้สุดท้าย
        })

# แปลง List เป็น DataFrame
research_df = pd.DataFrame(research_data)

if not research_df.empty:
    # คำนวณ RR จากกำไรที่ทำได้จริงๆ (Actual Result)
    research_df['RR_Ratio'] = round(research_df['Actual_Result'] / research_df['Box_Range'], 2)



=== สรุปสถิติเบื้องต้น ===
จำนวนรอบทั้งหมด (Total Trades): 312 ครั้ง
รอดจากการโดน SL (Wins): 197 ครั้ง
โดนตวัดกิน SL (Losses): 115 ครั้ง
Win Rate: 63.14%
